# Advanced tutorial: Land-cover change and water-quality trends

## Research question

**At Virginia STREAM stations, is increasing developed land cover resulting in increasing summer water temperature?**

## `stream_utils` functions used

- `StreamClient.find_sites()`
- `StreamClient.read_land_cover()`
- `StreamClient.read_water_quality()`
- `filter_season()`
- `resample_timeseries()`

In [ ]:
import numpy as np
import pandas as pd

from stream_utils.client import StreamClient
from stream_utils.temporal import filter_season, resample_timeseries

stream = StreamClient()

## 1. Select a manageable set of stations

In [ ]:
sites = stream.find_sites(states="Virginia", variables="WTemp_C")
site_ids = sites["STREAM_ID"].tolist()
print(f"Analyzing {len(site_ids)} stations")

## 2. Calculate annual developed land cover

In [ ]:
developed_columns = [
    "Developed Open Space", "Developed Low Intensity",
    "Developed Medium Intensity", "Developed High Intensity",
]
land_cover = stream.read_land_cover(site_ids, variables=developed_columns)
land_cover["year"] = pd.to_datetime(land_cover["DateTime"]).dt.year
land_cover["developed_fraction"] = land_cover[developed_columns].sum(axis=1)
annual_land_cover = land_cover[["STREAM_ID", "year", "developed_fraction"]]

## 3. Calculate annual summer temperature

In [ ]:
temperature = stream.read_water_quality(
    site_ids, "WTemp_C", quality="approved"
)
summer = filter_season(temperature, "summer")
monthly = resample_timeseries(summer, "WTemp_C", "monthly", "mean")
monthly["year"] = monthly["DateTime"].dt.year
annual_temperature = (
    monthly.groupby(["STREAM_ID", "year"], as_index=False)["WTemp_C"].mean()
)

## 4. Join by station and year, then estimate simple trends

In [ ]:
trends = annual_land_cover.merge(
    annual_temperature, on=["STREAM_ID", "year"], how="inner"
)

def slope(group, column):
    valid = group[["year", column]].dropna()
    return np.polyfit(valid["year"], valid[column], 1)[0] if len(valid) >= 2 else np.nan

station_trends = []
for stream_id, group in trends.groupby("STREAM_ID"):
    station_trends.append({
        "STREAM_ID": stream_id,
        "developed_trend_per_year": slope(group, "developed_fraction"),
        "temperature_trend_C_per_year": slope(group, "WTemp_C"),
        "matched_years": len(group),
    })
station_trends = pd.DataFrame(station_trends)
station_trends.sort_values("matched_years", ascending=False).head(10)

Please note that a trend association is exploratory and does not by itself establish that land-cover change caused a temperature change. Record length, missingness, climate, and other watershed differences should be considered.